# Post-Deploy Validation - R1 Modules

Validates that Memory, MLflow Telemetry, Dispatch, MCP, and API modules
are operational after an app deployment by querying OTel logs.
Run as part of the post-deploy validation workflow.

In [0]:
import time
from datetime import datetime, timedelta, timezone
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

APP_NAME = "mcp-hi-genie-orchestrator-dev"
OTEL_LOGS_TABLE = "hls_fde_dev.dev_matthew_giglia_hi_genie.hi_genie_otel_logs"
VALIDATION_WINDOW_MINUTES = 30

results = {}
cutoff = (datetime.now(timezone.utc) - timedelta(minutes=VALIDATION_WINDOW_MINUTES)).strftime("%Y-%m-%dT%H:%M:%S")

print(f"Validation started: {datetime.now(timezone.utc).isoformat()}")
print(f"App: {APP_NAME}")
print(f"OTel window: last {VALIDATION_WINDOW_MINUTES} minutes (since {cutoff})")

## Test 1: App Deployment Status

In [0]:
def test_app_deployment():
    """Verify the app has an active deployment in SUCCEEDED state."""
    app = w.apps.get(APP_NAME)
    deployment = app.active_deployment
    assert deployment is not None, "No active deployment found"
    status = deployment.status.state.value if deployment.status else "UNKNOWN"
    assert status == "SUCCEEDED", f"Deployment state is {status}, expected SUCCEEDED"
    print(f"  [PASS] Active deployment: {deployment.deployment_id} ({status})")
    return True

try:
    results["app_deployment"] = test_app_deployment()
except Exception as e:
    results["app_deployment"] = False
    print(f"  [FAIL] {e}")

## Test 2: Memory Module - Embedding Service

In [0]:
def test_memory_module():
    """Check OTel logs for memory module embedding service startup."""
    df = spark.sql(f"""
        SELECT CAST(body AS STRING) as msg, time FROM {OTEL_LOGS_TABLE}
        WHERE CAST(body AS STRING) LIKE '%[memory]%Embedding service started%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 5
    """)
    rows = df.collect()
    if rows:
        print(f"  [PASS] Memory embedding service started ({len(rows)} log entries)")
        print(f"         Latest: {str(rows[0]['msg'])[:100]}")
        return True

    # Check for the NOT started warning
    df_warn = spark.sql(f"""
        SELECT CAST(body AS STRING) as msg, time FROM {OTEL_LOGS_TABLE}
        WHERE CAST(body AS STRING) LIKE '%[memory]%NOT started%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 3
    """)
    warn_rows = df_warn.collect()
    if warn_rows:
        print(f"  [FAIL] Memory embedding NOT started (env vars missing):")
        print(f"         {str(warn_rows[0]['msg'])[:140]}")
        return False

    # Check for any memory initialization
    df_init = spark.sql(f"""
        SELECT CAST(body AS STRING) as msg, time FROM {OTEL_LOGS_TABLE}
        WHERE CAST(body AS STRING) LIKE '%[memory]%initialized%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 3
    """)
    init_rows = df_init.collect()
    if init_rows:
        print(f"  [WARN] Memory initialized but embedding service status unclear")
        return None  # inconclusive

    print(f"  [FAIL] No memory module logs found (app may not have restarted)")
    return False

try:
    results["memory_embedding"] = test_memory_module()
except Exception as e:
    results["memory_embedding"] = False
    print(f"  [FAIL] {e}")

## Test 3: MLflow Tracer - Initialization

In [0]:
def test_mlflow_tracer():
    """Check OTel logs for mlflow-tracer initialization."""
    df = spark.sql(f"""
        SELECT CAST(body AS STRING) as msg, time FROM {OTEL_LOGS_TABLE}
        WHERE CAST(body AS STRING) LIKE '%[mlflow-tracer]%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 5
    """)
    rows = df.collect()
    if not rows:
        print(f"  [WARN] No mlflow-tracer logs found")
        return None

    for r in rows:
        body = str(r['msg'])
        if 'initialized' in body.lower() or 'experiment' in body.lower():
            print(f"  [PASS] MLflow tracer initialized")
            print(f"         {body[:100]}")
            return True
        if '401' in body or '403' in body or 'permission' in body.lower():
            print(f"  [FAIL] MLflow tracer auth error:")
            print(f"         {body[:140]}")
            return False
        if 'disabled' in body.lower():
            print(f"  [WARN] MLflow tracer disabled by config")
            return None

    print(f"  [WARN] MLflow tracer logged but status unclear")
    return None

try:
    results["mlflow_tracer"] = test_mlflow_tracer()
except Exception as e:
    results["mlflow_tracer"] = False
    print(f"  [FAIL] {e}")

## Test 4: MCP Module - Tools Registered

In [0]:
def test_mcp_module():
    """Verify MCP tools were registered at startup."""
    df = spark.sql(f"""
        SELECT body, time FROM {OTEL_LOGS_TABLE}
        WHERE (body LIKE '%[mcp]%tool%' OR body LIKE '%MCP%registered%')
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 3
    """)
    rows = df.collect()
    if rows:
        print(f"  [PASS] MCP module tools registered")
        print(f"         {rows[0]['body'][:100]}")
        return True
    print(f"  [WARN] No MCP registration logs found")
    return None

try:
    results["mcp_tools"] = test_mcp_module()
except Exception as e:
    results["mcp_tools"] = False
    print(f"  [FAIL] {e}")

## Test 5: Dispatch Tick Evaluator

In [0]:
def test_dispatch_tick():
    """Verify dispatch tick evaluator is running."""
    df = spark.sql(f"""
        SELECT body, time FROM {OTEL_LOGS_TABLE}
        WHERE body LIKE '%[dispatch]%tick%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 3
    """)
    rows = df.collect()
    if rows:
        print(f"  [PASS] Dispatch tick active ({len(rows)} ticks logged)")
        return True

    # Broader check for any dispatch activity
    df_any = spark.sql(f"""
        SELECT body, time FROM {OTEL_LOGS_TABLE}
        WHERE body LIKE '%[dispatch]%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 3
    """)
    any_rows = df_any.collect()
    if any_rows:
        print(f"  [WARN] Dispatch logged but no tick entries")
        return None
    print(f"  [WARN] No dispatch logs found")
    return None

try:
    results["dispatch_tick"] = test_dispatch_tick()
except Exception as e:
    results["dispatch_tick"] = False
    print(f"  [FAIL] {e}")

## Test 6: Migrations Applied

In [0]:
def test_migrations():
    """Verify startup migrations completed."""
    df = spark.sql(f"""
        SELECT body, time FROM {OTEL_LOGS_TABLE}
        WHERE body LIKE '%[migrations]%'
          AND time >= '{cutoff}'
        ORDER BY time DESC LIMIT 5
    """)
    rows = df.collect()
    if rows:
        success = any('applied' in r['body'].lower() or 'complete' in r['body'].lower() or 'up to date' in r['body'].lower() for r in rows)
        if success:
            print(f"  [PASS] Migrations applied successfully")
            return True
        error = any('error' in r['body'].lower() or 'fail' in r['body'].lower() for r in rows)
        if error:
            print(f"  [FAIL] Migration errors detected:")
            for r in rows[:2]:
                print(f"         {r['body'][:100]}")
            return False
    print(f"  [WARN] No migration logs found")
    return None

try:
    results["migrations"] = test_migrations()
except Exception as e:
    results["migrations"] = False
    print(f"  [FAIL] {e}")

## Summary

In [0]:
print("\n" + "=" * 60)
print("POST-DEPLOY VALIDATION SUMMARY")
print("=" * 60)

critical_fail = False
for test_name, result in results.items():
    status = "PASS" if result is True else ("INCONCLUSIVE" if result is None else "FAIL")
    print(f"  [{status:>12}] {test_name}")
    if result is False:
        critical_fail = True

print("=" * 60)
if not critical_fail:
    print("All critical validations PASSED (inconclusive tests need manual review).")
else:
    print("CRITICAL FAILURES detected - review above.")

# Final assertion for job-based execution
assert not critical_fail, "Post-deploy validation failed - see results above"
print("\nValidation complete.")